# Приложение Д. Эффективная настройка параметров с помощью LoRA

In [1]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow", # Для предобученных весов OpenAI
        "pandas"      # Загрузка наборов данных
       ]
for p in pkgs:
    print(f"{p} версия: {version(p)}")

matplotlib версия: 3.10.9
numpy версия: 2.4.6
tiktoken версия: 0.13.0
torch версия: 2.12.0
tensorflow версия: 2.21.0
pandas версия: 3.0.3


## Д.1 Введение в LoRA

- Низкоранговая адаптация (Low-rank adaptation (LoRA)) — это метод машинного обучения, который модифицирует предобученную модель для лучшего соответствия конкретному, часто меньшему набору данных путём настройки лишь небольшого низкорангового подмножества параметров модели
- Этот подход важен, поскольку он позволяет эффективно дообучать большие модели на данных, специфичных для конкретной задачи, значительно снижая вычислительные затраты и время, необходимые для тонкой настройки

- Предположим, у нас есть большая весовая матрица $W$ для заданного слоя
- Во время обратного распространения мы изучаем матрицу $\Delta W$, которая содержит информацию о том, насколько мы хотим обновить исходные веса, чтобы минимизировать функцию потерь в процессе обучения
- При обычном обучении и тонкой настройке обновление весов определяется следующим образом:

$$W_{\text{updated}} = W + \Delta W$$

- Метод LoRA, предложенный [Hu et al.](https://arxiv.org/abs/2106.09685), предлагает более эффективную альтернативу вычислению обновлений весов $\Delta W$ путём изучения его аппроксимации, $\Delta W \approx AB$
- Другими словами, в LoRA мы имеем следующее, где $A$ и $B$ — две малые весовые матрицы:

$$W_{\text{updated}} = W + AB$$

- Рисунок ниже иллюстрирует эти формулы для полной тонкой настройки и LoRA рядом друг с другом

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-1.webp" width="800px">

- Изображения полной тонкой настройки и LoRA на рисунке выше выглядят немного иначе, чем формулы, которые были приведены ранее
- Это связано с дистрибутивным законом умножения матриц: нам не нужно складывать веса с обновлёнными весами, а можно держать их раздельно
- Например, если $x$ — это входные данные, то для обычной тонкой настройки мы можем записать следующее:

$$x (W+\Delta W) = x W + x \Delta W$$

- Аналогично, для LoRA мы можем записать следующее:

$$x (W+A B) = x W + x A B$$

- Тот факт, что мы можем держать весовые матрицы LoRA отдельно, делает LoRA особенно привлекательной
- На практике это означает, что нам вообще не нужно изменять веса предобученной модели, так как мы можем применять матрицы LoRA на лету
- После настройки набора данных и загрузки модели мы реализуем LoRA в коде, чтобы сделать эти концепции менее абстрактными

## Д.2 Подготовка набора данных

In [1]:
import requests
from pathlib import Path
import pandas as pd
from previous_chapters import (
    download_and_unzip_spam_data,
    create_balanced_dataset,
    random_split
)


url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Основной URL не сработал: {e}. Пробуем резервный URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

Файл скачан и сохранён как sms_spam_collection\SMSSpamCollection.tsv


In [2]:
import torch
import tiktoken
from previous_chapters import SpamDataset


tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", max_length=None, tokenizer=tokenizer)
val_dataset = SpamDataset("validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset = SpamDataset("test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)

In [3]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- В качестве проверочного шага мы итерируем по загрузчикам данных и проверяем, что каждый пакет содержит по 8 обучающих примеров, где каждый обучающий пример состоит из 120 токенов

In [4]:
print("Загрузчик обучающих данных:")
for input_batch, target_batch in train_loader:
    pass

print("Размеры входного пакета:", input_batch.shape)
print("Размеры пакета меток:", target_batch.shape)

Загрузчик обучающих данных:
Размеры входного пакета: torch.Size([8, 120])
Размеры пакета меток: torch.Size([8])


- Выведем общее количество пакетов в каждом наборе данных

In [5]:
print(f"{len(train_loader)} обучающих пакетов")
print(f"{len(val_loader)} валидационных пакетов")
print(f"{len(test_loader)} тестовых пакетов")

130 обучающих пакетов
19 валидационных пакетов
38 тестовых пакетов
